# NEWFIRM target submission (individual) to the AEON queue
Tomas Ahumada - tomas.ahumada@noirlab.edu

Last modified: Aug 2026


This code intends to be a tutrorial for NOIRLab-AEON users of the NOAO Extremely Wide Field Infrared Imager (NEWFIRM) mounted at the 4m V. M. Blanco telescope. The AEON queue is run by the Las Cumbres Observatory (LCO) Scheduler, thus the user requires an active account to access the LCO portal and generate a LCO key to submit requests to an active program.

Information about NEWFIRM can be found here: https://noirlab.edu/science/programs/ctio/instruments/newfirm

Information about LCO can be found here: https://observe.lco.global/

Once you have an active user in the LCO portal, you can find the API key here https://observe.lco.global/accounts/profile

# Outline
1. Get template request (json format) - this json file is modified and later sent to the LCO queue
3. Make the payload, modifying the json template.
4. Submit target individually. The target can have multiple filters.

# Exposure times

The recommended recipies for NEWFIRM:
| filter | exp time [s] | coadd |
|---|---|---|
| J | 20 | 2 |
| H | 10 | 3 |
| Ks | 10 | 3 |

In [3]:
import io
import json
import time
import urllib.request
from urllib.parse import urlparse

from astropy.time import Time
import numpy as np
import pandas as pd
import requests

In [4]:
# get example json

def get_example(example_path):
    if "github.com" in example_path and "/blob/" in example_path:
        example_path = example_path.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")

    if urlparse(example_path).scheme in ('http', 'https'):
        response = requests.get(example_path)

        if response.status_code == 200:
            return response.json()
        else:
            raise Exception(f"Failed to fetch file from URL. Status code: {response.status_code}")
    else:
        pass



In [5]:
template_json_url = "https://raw.githubusercontent.com/tahumada/newfirm-tda-tools/main/example.json"
get_example(template_json_url)

{'name': 'test',
 'proposal': '2025B-716366',
 'ipp_value': 1.05,
 'operator': 'SINGLE',
 'observation_type': 'NORMAL',
 'requests': [{'acceptability_threshold': 90,
   'configuration_repeats': 1,
   'optimization_type': 'TIME',
   'configurations': [{'type': 'EXPOSE',
     'instrument_type': 'BLANCO_NEWFIRM',
     'extra_params': {'dither_value': 80,
      'dither_sequence': '5-point',
      'detector_centering': 'det_1',
      'dither_sequence_random_offset': True},
     'instrument_configs': [{'exposure_count': 1,
       'exposure_time': '20',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '2',
        'sequence_repeats': 1,
        'offset_ra': 0,
        'offset_dec': 0},
       'optical_elements': {'filter': 'jx'}},
      {'exposure_count': 1,
       'exposure_time': '10',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '3',
        'sequence_repeats': 1,
        'offset_ra': 0,
        'offset_dec': 0

In [13]:
# Defining the variables you need to change:

PROPOSAL_ID  = 'HERE YOUR PROP_ID'
OBSERVATION_TYPE = 'NORMAL' # can be 'TIME_CRITICAL' or 'RAPID_RESPONSE'
WINDOW_START = '2026-08-01 17:32:00'
WINDOW_END   = '2026-09-01 17:32:00'
NAME = 'test'
RA = '15'
DEC = '-20'
FILTER_LIST = 'JHK'
SEQUENCE_REPEAT = 3
MAX_AIRMASS = 1.8
MIN_MOON = 30
DETECTOR_CENTERING = 'det_1'
DITHER_SEQUENCE = '5-point' # can be: '2x2', '3x3', '4x4', '5-point'
DITHER_VALUE = 80

variables = {
        "proposal": PROPOSAL_ID,
        'observation_type': OBSERVATION_TYPE,
        'maximum_airmass': MAX_AIRMASS,
        'minimum_lunar_distance': MIN_MOON,
        'detector_centering':  DETECTOR_CENTERING,
        'dither_value': DITHER_VALUE,
        'dither_sequence': DITHER_SEQUENCE,
        'sequence_repeats': SEQUENCE_REPEAT,
        'windows': [{'start': WINDOW_START, 'end': WINDOW_END}],
        'target': {'id': NAME,
                   'ra': RA,
                   'dec': DEC,
                   'filters': FILTER_LIST,
                  #  'exptime': EXPTIME_LIST, # this is fixed fro the template!
                   }
    }



data = get_example(template_json_url)

JX = data['requests'][0]['configurations'][0]['instrument_configs'][0]
HX = data['requests'][0]['configurations'][0]['instrument_configs'][1]
KX = data['requests'][0]['configurations'][0]['instrument_configs'][2]

req_config = data['requests'][0]['configurations'][0]

req_config['constraints']['max_airmass'] = variables['maximum_airmass']
req_config['constraints']['minimum_lunar_distance'] = variables['minimum_lunar_distance']

# Safely formatting the date (zero-padding months and days)
# now = Time.now().datetime
# date = f"{now.year}{now.month:02d}{now.day:02d}"
# data['name'] = f"{variables['target']['id']}_{date}"

data['name'] = variables['target']['id'] # name of your request
data['proposal'] = variables['proposal']
data['observation_type'] = variables['observation_type']
data['requests'][0]['windows'] = variables['windows']

req_config['target']['name'] = variables['target']['id'] # name of your target
req_config['target']['ra'] = str(variables['target']['ra'])
req_config['target']['dec'] = str(variables['target']['dec'])
req_config['extra_params']['detector_centering'] = variables['detector_centering']
req_config['extra_params']['dither_sequence'] = variables['dither_sequence']
req_config['extra_params']['dither_value'] = int(variables['dither_value'])

# base instrument configuration template
base_instrument_config = req_config['instrument_configs'][0]

# Reset filter configuration in the payload
req_config['instrument_configs'] = []

# loop through filters, copy, and append
if 'J' in variables['target']['filters']:
    JX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
    data['requests'][0]['configurations'][0]['instrument_configs'].append(JX)
if 'H' in variables['target']['filters']:
    HX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
    data['requests'][0]['configurations'][0]['instrument_configs'].append(HX)
if 'K' in variables['target']['filters']:
    KX['extra_params']['sequence_repeats'] = variables['sequence_repeats']
    data['requests'][0]['configurations'][0]['instrument_configs'].append(KX)

data

{'name': 'test',
 'proposal': 'newfirm_default',
 'ipp_value': 1.05,
 'operator': 'SINGLE',
 'observation_type': 'NORMAL',
 'requests': [{'acceptability_threshold': 90,
   'configuration_repeats': 1,
   'optimization_type': 'TIME',
   'configurations': [{'type': 'EXPOSE',
     'instrument_type': 'BLANCO_NEWFIRM',
     'extra_params': {'dither_value': 80,
      'dither_sequence': '5-point',
      'detector_centering': 'det_1',
      'dither_sequence_random_offset': True},
     'instrument_configs': [{'exposure_count': 1,
       'exposure_time': '20',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '2',
        'sequence_repeats': 3,
        'offset_ra': 0,
        'offset_dec': 0},
       'optical_elements': {'filter': 'jx'}},
      {'exposure_count': 1,
       'exposure_time': '10',
       'mode': 'fowler1',
       'rotator_mode': '',
       'extra_params': {'coadds': '3',
        'sequence_repeats': 3,
        'offset_ra': 0,
        'offset_dec'

In [14]:
LCO_TOKEN ='HERE YOUR LCO TOKEN'
requestpath  = "https://observe.lco.global/api/requestgroups/"

PROPOSAL_ID  = 'newfirm_default'
WINDOW_START = '2026-08-01 17:32:00'
WINDOW_END   = '2026-09-01 17:32:00'


lco_sent,lco_failed = [],[]
send = True



# send as test
if send:
  response = requests.post(
          requestpath,
          headers={"Authorization": f"Token {LCO_TOKEN}"},
          json=data,  # Make sure you use json!
      )

  if response.status_code == 400:
      print(variables['target']['id'], 'Failed (sending to queue)')
      lco_failed.append([variables['target']['id'],response.text])

  elif response.status_code == 201 or response.status_code == 200:
      lco_sent.append([variables['target']['id'],'sent!',response.json()['id']])



In [15]:
print('sources sent:', lco_sent)


sources sent: [['test', 'sent!', 2645157]]


In [16]:
print('sources failed:',lco_failed)

sources failed: []


In [18]:
ids = np.asarray(lco_sent).T[-1]
names = np.asarray(lco_sent).T[0]

for i,idlco in enumerate(ids):
    response = requests.post(
                f'https://observe.lco.global/api/requestgroups/{idlco}/cancel/',
                headers={"Authorization": f"Token {LCO_TOKEN}"},
            )

    if response.status_code == 200:
      print('request for:', names[i], 'cancelled\nid:', idlco)
    else:
      print('Cancel request failed for ', names[i])

request for: test cancelled
id: 2645157
